Bronze Layer --> Ingesting Data from API

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS job_market;

CREATE SCHEMA IF NOT EXISTS job_market.bronze;
CREATE SCHEMA IF NOT EXISTS job_market.silver;
CREATE SCHEMA IF NOT EXISTS job_market.gold;

In [0]:
import requests, json
from pyspark.sql import SparkSession 

# defining the spark session
spark = SparkSession.builder.appName('JobETL_Bronze').getOrCreate()

# defining the api key
API_KEY = 'API KEY'

roles = ['Data Engineer', 'SQL Developer', 'ETL Developer', 'Associate Data Engineer', 'Spark Engineer', 'Databricks Engieer', 'Azure Data Engineer']
location = ['India']
all_jobs = []

for role in roles:
    params = {
        'engine': 'google_jobs',
        'q': role,
        'location': location,
        'api_key': API_KEY
    }

    res = requests.get('https://serpapi.com/search.json', params=params)
    print("✅Reading 🔴LIVE Data from Google Jobs API")
    jobs = res.json().get('jobs_results', [])

    for job in jobs:
        job['search_role'] = role
    
    all_jobs.extend(jobs)

all_jobs_df = spark.createDataFrame(all_jobs)
all_jobs_df.write.format('delta').option('mergeSchema', 'true').mode('append').saveAsTable('job_market.bronze.raw_jobs')
print("✅Data Written to Bronze Layer")

In [0]:
%sql
-- select * from job_market.bronze.raw_jobs;